# Publication notes

This is the original exploratory workflow with saved outputs removed and paths made configurable. Collection and analytical logic is retained. Source websites and identifiers may change; notebooks 01–02 are an inspected research workflow, not a guaranteed unattended scraper. Review collection permissions and source-specific settings before recollection. Do not run against your only copy of historical data: original export cells can overwrite files.

Run from a clean kernel in order. Data is local and excluded from GitHub. Historical saved review text and reviewer details are not published as notebook outputs.


In [ ]:
from pathlib import Path
import os

# Configure KBEAUTY_PROJECT_ROOT to keep data outside the cloned repository.
_working_dir = Path.cwd().resolve()
_default_root = _working_dir.parent if _working_dir.name == "notebooks" else _working_dir
PROJECT_ROOT = Path(os.environ.get("KBEAUTY_PROJECT_ROOT", str(_default_root))).expanduser().resolve()
# Original collection cells use relative data/ paths.
os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)


In [ ]:
import requests

url = "https://beautyofjoseon.com/products/ginseng-essence-water"

response = requests.get(url)

print("Status code:", response.status_code)

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, "html.parser")

print(soup.title)

In [ ]:
import json

json_scripts = soup.find_all("script", type="application/ld+json")

print("Number of JSON-LD blocks:", len(json_scripts))

for i, script in enumerate(json_scripts):
    print("\n--- BLOCK", i, "---")
    print(script.get_text(strip=True)[:500])

In [ ]:
product_data = json.loads(json_scripts[3].string)

print(json.dumps(product_data, indent=2))

In [ ]:
product_data

In [ ]:
product_record = {
    "product_id": "BOJ_TONER_01",
    "brand": product_data["brand"]["name"],
    "product_name": product_data["name"],
    "segment": "Toner / Essence",
    "sku": product_data["sku"],
    "current_price": product_data["offers"][0]["price"],
    "currency": product_data["offers"][0]["priceCurrency"],
    "description": product_data["description"].strip(),
    "product_url": product_data["url"]
}

product_record

In [ ]:
page_text = soup.get_text(" ", strip=True)

keyword = "ingredients"
position = page_text.lower().find(keyword)

print("Position:", position)
print(page_text[position:position + 1500])

In [ ]:
html = response.text

keyword = "Niacinamide"
position = html.lower().find(keyword.lower())

print("Position:", position)
print(html[position - 500:position + 1500])

In [ ]:
import re

matches = [
    match.start()
    for match in re.finditer("niacinamide", html, re.IGNORECASE)
]

print("Number of occurrences:", len(matches))
print("Positions:", matches)

In [ ]:
print(html[899500:903000])

In [ ]:
marker = "customMetafields:"
start = html.find(marker)

print("Start position:", start)
print(html[start:start + 1000])

In [ ]:
marker = "customMetafields:"
start = html.find(marker) + len(marker)

decoder = json.JSONDecoder()
custom_metafields, end = decoder.raw_decode(html[start:].lstrip())

print("Key ingredients:", custom_metafields.get("key_ingredients"))
print("Key benefits:", custom_metafields.get("key_benefits"))
print("Skin concern:", custom_metafields.get("skin_concern"))
print("\nFull ingredients:")
print(custom_metafields.get("all_ingredients")[0])

In [ ]:
product_record.update({
    "key_ingredients": custom_metafields.get("key_ingredients"),
    "key_benefits": custom_metafields.get("key_benefits"),
    "skin_concern": custom_metafields.get("skin_concern"),
    "ingredients": custom_metafields.get("all_ingredients")[0]
})

product_record

In [ ]:
import pandas as pd

products_df = pd.DataFrame([product_record])

products_df

In [ ]:
reviews_url = "https://api.okendo.io/v1/stores/121d630c-0612-42f9-8f88-70cb5ef6a19b/products/shopify-6805137195173/reviews"

params = {
    "limit": 5,
    "orderBy": "rating desc",
    "locale": "en"
}

reviews_response = requests.get(reviews_url, params=params)

print("Status code:", reviews_response.status_code)

In [ ]:
reviews_data = reviews_response.json()

print(type(reviews_data))
print(reviews_data.keys())

In [ ]:
print("Number of reviews returned:", len(reviews_data["reviews"]))

first_review = reviews_data["reviews"][0]

print("\nReview fields:")
print(first_review.keys())

In [ ]:
import json

print(json.dumps(first_review, indent=2))

In [ ]:
# Extract reviewer attributes
reviewer_attributes = {
    attribute["title"]: attribute["value"]
    for attribute in first_review["reviewer"]["attributes"]
}

review_record = {
    "review_id": first_review["reviewId"],
    "product_id": "BOJ_TONER_01",
    "review_date": first_review["dateCreated"],
    "rating": first_review["rating"],
    "review_title": first_review["title"],
    "review_text": first_review["body"],
    "reviewer_name": first_review["reviewer"]["displayName"],
    "age": reviewer_attributes.get("How old are you?"),
    "skin_type": reviewer_attributes.get("What is your skin type?"),
    "skin_concerns": reviewer_attributes.get("What are your skin concerns?"),
    "verified": first_review["reviewer"]["isVerified"],
    "recommended": first_review["isRecommended"],
    "incentivized": first_review["isIncentivized"],
    "helpful_count": first_review["helpfulCount"],
    "unhelpful_count": first_review["unhelpfulCount"],
    "language": first_review["languageCode"]
}

review_record

In [ ]:
print(reviews_data["nextUrl"])

In [ ]:
print(reviews_data["nextUrl"])

In [ ]:
base_url = "https://api.okendo.io/v1"

current_url = reviews_url
current_params = {
    "limit": 5,
    "orderBy": "rating desc",
    "locale": "en"
}

test_reviews = []

for page in range(2):
    
    response_page = requests.get(
        current_url,
        params=current_params
    )
    
    response_page.raise_for_status()
    page_data = response_page.json()
    
    test_reviews.extend(page_data["reviews"])
    
    print(
        f"Page {page + 1}: "
        f"{len(page_data['reviews'])} reviews collected"
    )
    
    next_url = page_data.get("nextUrl")
    
    if not next_url:
        break
    
    current_url = base_url + next_url
    
    # nextUrl already contains the query parameters
    current_params = None

print("\nTotal reviews collected:", len(test_reviews))

In [ ]:
def collect_reviews(reviews_url, product_id, max_reviews=300):
    
    base_url = "https://api.okendo.io/v1"
    
    current_url = reviews_url
    current_params = {
        "limit": 25,
        "locale": "en"
    }
    
    collected_reviews = []
    
    while len(collected_reviews) < max_reviews:
        
        response = requests.get(
            current_url,
            params=current_params
        )
        
        response.raise_for_status()
        data = response.json()
        
        reviews = data.get("reviews", [])
        
        if not reviews:
            break
        
        # Add our own relational product ID
        for review in reviews:
            review["relational_product_id"] = product_id
        
        collected_reviews.extend(reviews)
        
        print(
            f"Collected {min(len(collected_reviews), max_reviews)} "
            f"of {max_reviews} reviews"
        )
        
        next_url = data.get("nextUrl")
        
        if not next_url:
            break
        
        current_url = base_url + next_url
        current_params = None
    
    return collected_reviews[:max_reviews]

In [ ]:
ginseng_reviews = collect_reviews(
    reviews_url=reviews_url,
    product_id="BOJ_TONER_01",
    max_reviews=300
)

print("\nFinal number of reviews:", len(ginseng_reviews))

In [ ]:
def clean_review(review):
    
    # Extract reviewer information safely
    reviewer = review.get("reviewer", {})
    attributes = reviewer.get("attributes", [])
    
    # Convert reviewer attributes into a dictionary
    reviewer_attributes = {
        attribute["title"]: attribute["value"]
        for attribute in attributes
    }
    
    return {
        "review_id": review.get("reviewId"),
        "product_id": review.get("relational_product_id"),
        "review_date": review.get("dateCreated"),
        "rating": review.get("rating"),
        "review_title": review.get("title"),
        "review_text": review.get("body"),
        "reviewer_name": reviewer.get("displayName"),
        "age": reviewer_attributes.get("How old are you?"),
        "skin_type": reviewer_attributes.get("What is your skin type?"),
        "skin_concerns": reviewer_attributes.get("What are your skin concerns?"),
        "verified": reviewer.get("isVerified"),
        "recommended": review.get("isRecommended"),
        "incentivized": review.get("isIncentivized"),
        "helpful_count": review.get("helpfulCount"),
        "unhelpful_count": review.get("unhelpfulCount"),
        "language": review.get("languageCode")
    }

In [ ]:
clean_ginseng_reviews = [
    clean_review(review)
    for review in ginseng_reviews
]

reviews_df = pd.DataFrame(clean_ginseng_reviews)

reviews_df.head()

In [ ]:
print("Dataset shape:", reviews_df.shape)

print("\nColumns:")
print(reviews_df.columns.tolist())

In [ ]:
reviews_df.isnull().sum()

In [ ]:
reviews_df["rating"].value_counts().sort_index()

In [ ]:
reviews_df["review_date"] = pd.to_datetime(
    reviews_df["review_date"],
    errors="coerce"
)

print("Newest review:", reviews_df["review_date"].max())
print("Oldest review:", reviews_df["review_date"].min())

In [ ]:
print(
    "Duplicate review IDs:",
    reviews_df["review_id"].duplicated().sum()
)

In [ ]:
review_skin_concerns_df = (
    reviews_df[["review_id", "skin_concerns"]]
    .dropna(subset=["skin_concerns"])
    .explode("skin_concerns")
    .rename(columns={"skin_concerns": "skin_concern"})
    .reset_index(drop=True)
)

review_skin_concerns_df.head(10)

In [ ]:
review_skin_concerns_df = (
    review_skin_concerns_df
    .drop_duplicates(subset=["review_id", "skin_concern"])
    .reset_index(drop=True)
)

print("Bridge table rows:", len(review_skin_concerns_df))
print(
    "Duplicate review-concern pairs:",
    review_skin_concerns_df.duplicated(
        subset=["review_id", "skin_concern"]
    ).sum()
)

review_skin_concerns_df.head(10)

In [ ]:
reviews_clean_df = reviews_df.drop(
    columns=["skin_concerns"]
).copy()

print("Reviews table shape:", reviews_clean_df.shape)
print("\nColumns:")
print(reviews_clean_df.columns.tolist())

reviews_clean_df.head()

In [ ]:
product_data = {
    "product_id": "BOJ_TONER_01",
    "brand": "Beauty of Joseon",
    "product_name": "Ginseng Essence Water",
    "segment": "Toner / Essence",
    "sku": "82BS002",
    "current_price": 13.50,
    "currency": "USD",
    "description": (
        "A barrier-boosting essence packed with ginseng water and "
        "niacinamide for lasting hydration, nourishment, and a glow boost."
    ),
    "key_ingredients": "Ginseng + Niacinamide",
    "key_benefits": "Long Lasting Hydration & Skin Calming Effect",
    "skin_concern": "Uneven Skin Tone + Dryness",
    "product_url": "https://beautyofjoseon.com/products/ginseng-essence-water"
}

products_df = pd.DataFrame([product_data])

products_df

In [ ]:
ingredient_list = [
    "Water",
    "Butylene Glycol",
    "Glycerin",
    "Propanediol",
    "Niacinamide",
    "1,2-Hexanediol",
    "Panax Ginseng Root Water",
    "Hydroxyacetophenone",
    "Glyceryl Glucoside",
    "Panthenol",
    "Xanthan Gum",
    "Allantoin",
    "Dipotassium Glycyrrhizate",
    "Adenosine",
    "Dextrin",
    "Theobroma Cacao (Cocoa) Extract",
    "Disodium EDTA",
    "Glucose",
    "Panax Ginseng Callus Culture Extract",
    "Panax Ginseng Root Extract",
    "Panax Ginseng Berry Extract",
    "Ethylhexylglycerin",
    "Lactobacillus/Panax Ginseng Root Ferment Filtrate",
    "Sodium Hyaluronate"
]

ingredients_df = pd.DataFrame({
    "product_id": "BOJ_TONER_01",
    "ingredient_position": range(1, len(ingredient_list) + 1),
    "ingredient_name": ingredient_list
})

print("Number of ingredients:", len(ingredients_df))
ingredients_df

In [ ]:
print("PRODUCTS")
print("Rows:", len(products_df))
print("Unique product IDs:", products_df["product_id"].nunique())

print("\nREVIEWS")
print("Rows:", len(reviews_clean_df))
print("Unique review IDs:", reviews_clean_df["review_id"].nunique())
print("Unique product IDs:", reviews_clean_df["product_id"].nunique())

print("\nINGREDIENTS")
print("Rows:", len(ingredients_df))
print("Unique products:", ingredients_df["product_id"].nunique())

print("\nREVIEW SKIN CONCERNS")
print("Rows:", len(review_skin_concerns_df))
print("Unique reviews:", review_skin_concerns_df["review_id"].nunique())

In [ ]:
# Reviews whose product_id does not exist in Products
orphan_reviews = reviews_clean_df[
    ~reviews_clean_df["product_id"].isin(products_df["product_id"])
]

# Ingredients whose product_id does not exist in Products
orphan_ingredients = ingredients_df[
    ~ingredients_df["product_id"].isin(products_df["product_id"])
]

# Skin concerns whose review_id does not exist in Reviews
orphan_concerns = review_skin_concerns_df[
    ~review_skin_concerns_df["review_id"].isin(reviews_clean_df["review_id"])
]

print("Orphan reviews:", len(orphan_reviews))
print("Orphan ingredients:", len(orphan_ingredients))
print("Orphan review concerns:", len(orphan_concerns))

In [ ]:
from pathlib import Path

output_folder = Path("data/processed/test_product")
output_folder.mkdir(parents=True, exist_ok=True)

print("Saving files to:", output_folder.resolve())

In [ ]:
products_df.to_csv(
    output_folder / "products.csv",
    index=False
)

reviews_clean_df.to_csv(
    output_folder / "reviews.csv",
    index=False
)

ingredients_df.to_csv(
    output_folder / "ingredients.csv",
    index=False
)

review_skin_concerns_df.to_csv(
    output_folder / "review_skin_concerns.csv",
    index=False
)

print("All tables saved successfully!")

In [ ]:
for file in output_folder.glob("*.csv"):
    print(file.name)